<a href="https://colab.research.google.com/github/Premkumarlodhi/SOC-CV/blob/main/sushi-pizza-steak%20classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 🚀 Full PyTorch Image Classification Template (Pizza/Steak/Sushi)
import os
import torch
import requests
import zipfile
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch import nn

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Download and prepare 20% dataset
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi_20_percent"

if not image_path.is_dir():
    print(f"Creating dataset folder and downloading...")
    image_path.mkdir(parents=True, exist_ok=True)
    with open(data_path / "pizza_steak_sushi_20_percent.zip", "wb") as f:
        r = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip")
        f.write(r.content)
    with zipfile.ZipFile(data_path / "pizza_steak_sushi_20_percent.zip", "r") as zip_ref:
        zip_ref.extractall(image_path)

# Data loaders
def get_dataloaders(train_dir, test_dir, batch_size=32, image_size=(64,64)):
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor()
    ])
    train_ds = ImageFolder(root=train_dir, transform=transform)
    test_ds = ImageFolder(root=test_dir, transform=transform)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(test_ds, batch_size=batch_size),
        train_ds.classes,
        train_ds.class_to_idx
    )

train_loader, test_loader, classes, class_to_idx = get_dataloaders(
    image_path / "train", image_path / "test"
)
# Model
class Model0(nn.Module):
    def __init__(self, input_shape=(3,64,64), hidden_units=200, output_classes=3):
        super().__init__()
        in_features = input_shape[0] * input_shape[1] * input_shape[2]
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, output_classes)
        )
    def forward(self, x):
        return self.net(x)

model = Model0().to(device)

# Training and testing steps
def train_step(model, dataloader, loss_fn, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        preds = model(X)
        loss = loss_fn(preds, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct += (preds.argmax(1) == y).sum().item()
        total += X.size(0)
    return total_loss / total, correct / total

def test_step(model, dataloader, loss_fn):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            preds = model(X)
            loss = loss_fn(preds, y)
            total_loss += loss.item() * X.size(0)
            correct += (preds.argmax(1) == y).sum().item()
            total += X.size(0)
    return total_loss / total, correct / total

# Training loop
def train(model, train_loader, test_loader, optimizer, loss_fn=nn.CrossEntropyLoss(), epochs=20):
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    for epoch in tqdm(range(epochs), desc="Epochs"):
        train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer)
        test_loss, test_acc = test_step(model, test_loader, loss_fn)
        print(f"Epoch {epoch+1}: "
              f"Train loss {train_loss:.4f}, acc {train_acc:.4f} | "
              f"Test loss {test_loss:.4f}, acc {test_acc:.4f}")
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
    return results

# Train the model
torch.manual_seed(42)
torch.cuda.manual_seed(42)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train(model, train_loader, test_loader, optimizer, epochs=20)




Using device: cpu
Creating dataset folder and downloading...


Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1: Train loss 2.3710, acc 0.3667 | Test loss 1.3583, acc 0.4667
Epoch 2: Train loss 1.2241, acc 0.4911 | Test loss 1.3282, acc 0.5667
Epoch 3: Train loss 1.1643, acc 0.5689 | Test loss 1.4627, acc 0.4067
Epoch 4: Train loss 1.2250, acc 0.4733 | Test loss 1.1675, acc 0.4733
Epoch 5: Train loss 0.9926, acc 0.5356 | Test loss 0.9959, acc 0.5667
Epoch 6: Train loss 0.8499, acc 0.6044 | Test loss 0.9184, acc 0.5600
Epoch 7: Train loss 0.7843, acc 0.6378 | Test loss 1.0037, acc 0.5400
Epoch 8: Train loss 0.7501, acc 0.6756 | Test loss 0.9023, acc 0.5933
Epoch 9: Train loss 0.6997, acc 0.7067 | Test loss 1.0023, acc 0.5267
Epoch 10: Train loss 0.7310, acc 0.6600 | Test loss 0.9925, acc 0.5400
Epoch 11: Train loss 0.7296, acc 0.6733 | Test loss 0.8808, acc 0.6200
Epoch 12: Train loss 0.6085, acc 0.7422 | Test loss 0.9142, acc 0.5933
Epoch 13: Train loss 0.5891, acc 0.7711 | Test loss 1.0349, acc 0.5200
Epoch 14: Train loss 0.6409, acc 0.7600 | Test loss 1.4130, acc 0.5200
Epoch 15: Train

{'train_loss': [2.3709608528349135,
  1.2240806516011555,
  1.1643328306410048,
  1.2249521836307313,
  0.9925618945227729,
  0.849892299440172,
  0.7842878111203512,
  0.7500763730208079,
  0.6996539374192555,
  0.7309846435652839,
  0.7296085675557454,
  0.6085451579093933,
  0.5890764180819194,
  0.6408941748407152,
  0.821601333618164,
  0.6418493286768595,
  0.7541476302676731,
  0.6452317884233263,
  0.52724264409807,
  0.5334396457672119],
 'train_acc': [0.36666666666666664,
  0.4911111111111111,
  0.5688888888888889,
  0.47333333333333333,
  0.5355555555555556,
  0.6044444444444445,
  0.6377777777777778,
  0.6755555555555556,
  0.7066666666666667,
  0.66,
  0.6733333333333333,
  0.7422222222222222,
  0.7711111111111111,
  0.76,
  0.6911111111111111,
  0.7288888888888889,
  0.66,
  0.7288888888888889,
  0.8,
  0.8],
 'test_loss': [1.358299249013265,
  1.3282066027323405,
  1.4626515273253122,
  1.1675344562530519,
  0.995887173016866,
  0.9184410643577575,
  1.0037406619389853,


In [5]:
custom_image_path = "/content/sushi.webp"

# Load and transform
custom_img = Image.open(custom_image_path).convert("RGB")
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])
custom_tensor = transform(custom_img).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.inference_mode():
    pred = model(custom_tensor).argmax(dim=1).item()
print(f"Prediction: {classes[pred]} (index {pred})")

Prediction: sushi (index 2)
